# Phase 5: Model Training — Transfer Learning (Frozen)

In this phase, we use **ResNet34**, which is already pre-trained on ImageNet. Instead of training the entire network from scratch (which requires massive datasets and computing resources), we will:

1. **Freeze** the existing convolutional layers (feature extractors).
2. **Replace** the final linear classification head with a new 2-class output (`open` vs `closed`).
3. **Train** only this newly appended output layer on our carabiner dataset for 10 epochs.

### 1. Setup & Data Loading

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import time
import copy
from pathlib import Path
import os

# Set device 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

# Paths (Using the processed data generated in Phase 4)
PROCESSED_DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

mean = [0.485, 0.456, 0.406] 
std = [0.229, 0.224, 0.225]

# Redefine Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}

# Dataloaders
image_datasets = {x: datasets.ImageFolder(root=PROCESSED_DATA_DIR / x, transform=data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=32, shuffle=(x == 'train'), num_workers=0) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Classes: {class_names}")
print(f"Dataset Sizes: {dataset_sizes}")

### 2. Model Configuration

In [ ]:
from torchvision.models import ResNet34_Weights

# 1. Load Pre-trained ResNet34
model = models.resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)

# 2. Freeze the base layers
for param in model.parameters():
    param.requires_grad = False

# 3. Replace the classification head
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

model = model.to(device)

# Criteria and Optimizer
criterion = nn.CrossEntropyLoss()

# Note: We are specifically passing in ONLY the `model.fc.parameters()` to the optimizer 
# because the rest of the model is frozen and doesn't require gradient tracking.
optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3)

### 3. Training Loop Defintion

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  
            else:
                model.eval()   

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model, history

### 4. Execute Training

In [ ]:
model_frozen, training_history = train_model(model, criterion, optimizer, num_epochs=10)

# Save the weights to our models directory 
save_path = MODELS_DIR / 'best_frozen_resnet34.pth'
torch.save(model_frozen.state_dict(), save_path)
print(f"\nSaved best frozen model weights to: {save_path}")